# Record Reachy joint limits
Move the limp arm by hand to the furthest point you trust for each joint. The lowest and highest angle seen, pulled inward by `MARGIN` degrees, get merged into **`joint_limits.json`**. Joints and arms already in that file are kept, so you can record one arm or one joint at a time.

Other code reads the limits through `joint_limits.py` (`check_limits`, `safe_goto`).

Run top to bottom. Angles are in degrees.

In [ ]:
PART = 'r_arm'      # 'r_arm', 'l_arm' or 'head'
JOINTS = None       # None = every joint in PART, or e.g. ['r_shoulder_pitch']
MARGIN = 3.0        # degrees to pull each limit inward
HOST = None         # None = findIP.get_reachy_ip(), or e.g. '169.254.50.2'

## 1. Connect

In [ ]:
import json, time
from datetime import datetime
from IPython.display import clear_output
from reachy_sdk import ReachySDK
from joint_limits import LIMITS_FILE

if HOST is None:
    from findIP import get_reachy_ip
    HOST = get_reachy_ip()
reachy = ReachySDK(host=HOST)

joints = list(getattr(reachy, PART).joints.values())
if JOINTS:
    unknown = set(JOINTS) - {j.name for j in joints}
    assert not unknown, f'not in {PART}: {sorted(unknown)}. Available: {[j.name for j in joints]}'
    joints = [j for j in joints if j.name in JOINTS]
print('recording:', [j.name for j in joints])

## 2. Make the arm limp
**Hold the arm first.** It drops as soon as this cell runs.

In [ ]:
reachy.turn_off(PART)

## 3. Record
Run the cell, then move each joint slowly to its safe extremes in both directions. Press **Interrupt (■)** to stop. If a joint's angle jumps by more than 180° between two readings, it wrapped around ±180°, and that joint gets marked unusable instead of saved with a wrong range.

In [ ]:
WRAP_JUMP_DEG = 180
seen = {j.name: [j.present_position] * 2 for j in joints}
prev = {j.name: j.present_position for j in joints}
wrapped = set()

last_print = 0.0
try:
    while True:
        for j in joints:
            p = j.present_position
            if abs(p - prev[j.name]) > WRAP_JUMP_DEG:
                wrapped.add(j.name)
            prev[j.name] = p
            lo, hi = seen[j.name]
            seen[j.name] = [min(lo, p), max(hi, p)]
        if time.time() - last_print > 0.3:
            last_print = time.time()
            clear_output(wait=True)
            print('Recording... Interrupt (■) to stop.\n')
            for j in joints:
                lo, hi = seen[j.name]
                flag = '  WRAPPED' if j.name in wrapped else ''
                print(f'{j.name:22s} now {prev[j.name]:8.1f}   min {lo:8.1f}   max {hi:8.1f}{flag}')
        time.sleep(0.02)
except KeyboardInterrupt:
    print('\nstopped.')

## 4. Review

In [ ]:
stamp = datetime.now().isoformat(timespec='seconds')
new = {}
for name, (lo, hi) in seen.items():
    lo_m, hi_m = lo + MARGIN, hi - MARGIN
    if lo_m >= hi_m:
        print(f'skip {name}: range {lo:.1f}..{hi:.1f} too small for margin {MARGIN} (did you move it?)')
        continue
    new[name] = {
        'min': round(lo_m, 2), 'max': round(hi_m, 2),
        'raw_min': round(lo, 2), 'raw_max': round(hi, 2),
        'margin': MARGIN, 'wrapped': name in wrapped, 'recorded': stamp,
    }
    flag = '  (WRAPPED, unusable)' if name in wrapped else ''
    print(f'{name:22s} {lo_m:8.1f} .. {hi_m:8.1f}{flag}')

## 5. Save
Set `SAVE = True` once the numbers above look right.

In [ ]:
SAVE = False

if SAVE and new:
    data = json.loads(LIMITS_FILE.read_text()) if LIMITS_FILE.exists() else {}
    data.update(new)
    LIMITS_FILE.write_text(json.dumps(dict(sorted(data.items())), indent=2) + '\n')
    print(f'saved {len(new)} joint(s) to {LIMITS_FILE}')
else:
    print('not saved')

## 6. Done
The arm is still limp. Hold it and run this to let it down gently, or use `reachy.turn_on(PART)` to make it stiff where it is.

In [ ]:
reachy.turn_off_smoothly(PART)